<a href="https://colab.research.google.com/github/jetsonai/StartClassBeginner/blob/main/MachineLearning/ml_kiosk_02_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2강. 머신러닝 데이터 기초
1년 치 주문 데이터 다루기 · 정답 노트

> 0. GitHub에서 보고 있다면 위의 <b>Open in Colab</b> 버튼을 눌러 Colab에서 여세요.
> 1. Colab에서 열었다면 먼저 <b>[파일] → [Drive에 사본 저장]</b> 메뉴로 내 드라이브에 복사하세요. 그래야 수정한 내용이 저장됩니다.
> 2. 셀은 <b>위에서부터 순서대로</b> 실행하세요. 연결이 끊겼다면 처음부터 다시 실행하면 됩니다.


## 실습 준비
그래프 한글 설정, 데이터 주소, 메뉴 이름표가 들어 있어요. <b>가장 먼저 실행</b>하세요. 연결이 끊겼다면 이 셀부터 다시 실행하면 됩니다.

In [ ]:
!pip install -q koreanize-matplotlib
import koreanize_matplotlib   # 그래프 한글 깨짐 방지

# 실습 데이터 주소 (1년 치 주문 기록)
URL = "https://raw.githubusercontent.com/jetsonai/StartClassBeginner/main/MachineLearning/kiosk_sales.csv"

# 메뉴 열 이름(영어) → 그래프에 보일 한글 이름
names = {"americano": "아메리카노", "ice_americano": "아이스아메리카노", "latte": "카페라테",
         "oat_latte": "오트라테", "cheesecake": "치즈케이크", "sandwich": "샌드위치", "fruit_cup": "과일컵"}
menu_cols = list(names)   # 메뉴 열 이름 목록
print(menu_cols)

## S10. 리스트에 2를 곱하면?
리스트는 계산용이 아니에요. 곱하면 장바구니가 반복됩니다. (리스트는 파이썬 1강)

In [ ]:
prices = [2000, 3000, 4500]   # 아메리카노, 카페라테, 치즈케이크
print(prices * 2)

## S12. NumPy 배열: 한 번에 계산
`np.array`로 바꾸면 계산이 칸마다 한 번에 적용돼요.

In [ ]:
import numpy as np

p = np.array(prices)
print(p * 2)          # 모두 두 배
print(p + 500)        # 모두 500원 인상
print(p.sum(), p.max())

## S14. 2차원 배열: 날짜 × 메뉴 판매표
`shape`는 (행, 열). `axis=0`은 세로로(메뉴별), `axis=1`은 가로로(날짜별) 계산해요.

In [ ]:
sales = np.array([[30, 45, 12],    # 월
                  [28, 40, 10],    # 화
                  [35, 20, 25]])   # 토
print(sales.shape)
print(sales.sum(axis=0))   # 메뉴별 합계 ↓
print(sales.sum(axis=1))   # 날짜별 합계 →

## S18. 1년 치 주문 불러오기
`read_csv`로 CSV 파일을 표(DataFrame)로 읽어요. `head()`는 앞 5줄을 보여 줍니다.

In [ ]:
import pandas as pd

df = pd.read_csv(URL)   # URL은 실습 준비 셀에 있어요
print(df.shape)
df.head()

## S20. 열 꺼내기와 요약
`df["열 이름"]`으로 메뉴판처럼 이름으로 꺼내요. count를 잘 보세요!

In [ ]:
print(round(df["ice_americano"].mean(), 1))   # 하루 평균
df["temp"].describe()

## S22. 빈칸(결측치) 찾고 채우기
`NaN`은 비어 있다는 뜻이에요. `interpolate()`는 앞뒤 날 사이 값으로 채워 줍니다.
> 이 셀을 실행해야 아래 결과가 슬라이드와 같아져요.

In [ ]:
print(df["temp"].isna().sum())       # 빈칸 개수

df["temp"] = df["temp"].interpolate()  # 앞뒤 날 사이 값으로 채우기
print(df["temp"].isna().sum())

## S24. 조건으로 고르기: 더운 날만
`df[조건]`은 조건이 True인 행만 골라요. (비교 연산자는 파이썬 2강-1)

In [ ]:
hot = df[df["temp"] >= 30]       # 30도 이상인 날만

print(len(hot))
print(round(hot["ice_americano"].mean(), 1))   # 더운 날 평균
print(round(df["ice_americano"].mean(), 1))    # 전체 평균

## S26. 그룹별 평균: 주말엔 뭐가 잘 팔릴까?
`isin`으로 토·일이면 True인 새 열을 만들고, `groupby`로 같은 값끼리 묶어 평균을 내요.

In [ ]:
df["weekend"] = df["weekday"].isin(["토", "일"])   # 새 열 만들기
df.groupby("weekend")[["latte", "cheesecake"]].mean().round(1)

## S29. 메뉴별 1년 판매량: 막대그래프
`menu_cols`와 `names`는 실습 준비 셀에 있어요.

In [ ]:
total = df[menu_cols].sum()     # 메뉴별 1년 합계

total.rename(names).plot(kind="bar", rot=45,
                         title="메뉴별 1년 판매량")

## S31. 기온과 아이스 아메리카노: 산점도
점 하나가 하루예요. 어떤 모양인지 보세요.

In [ ]:
df.plot(kind="scatter",
        x="temp", y="ice_americano",
        xlabel="최고기온(℃)",
        ylabel="아이스 아메리카노(잔)")

## S33. [선택] 월별 판매 추이: 선 그래프
시간에 따라 변하는 데이터(시계열)예요. 6강에서 다시 만나요.

In [ ]:
monthly = df.groupby("month")["ice_americano"].sum()

monthly.plot(marker="o", xlabel="월",
             title="월별 아이스 아메리카노")

## S36. 특성(X)과 정답(y) 나누기
`X`는 단서가 담긴 표(대괄호 두 겹), `y`는 맞히고 싶은 열 하나(대괄호 한 겹)예요. 3강에서 이 X와 y로 모델을 학습시킵니다.

In [ ]:
X = df[["temp", "rain"]]     # 특성: 대괄호 두 겹 → 표
y = df["ice_americano"]      # 정답: 대괄호 한 겹 → 열 하나

print(X.shape, y.shape)
X.head(3)

## 복습 미션: 따뜻한 아메리카노는?
1) 비 오는 날(`rain` = 1)과 안 오는 날의 아메리카노 평균, 2) 기온과 아메리카노 산점도를 그려 보세요.
아래처럼 나오고, 산점도가 <b>오른쪽 아래로 내려가면</b> 성공!
```
rain
0    41.0
1    36.8
```

In [ ]:
# 1) 비 오는 날과 안 오는 날의 아메리카노 평균
df.groupby("rain")["americano"].mean().round(1)

In [ ]:
# 2) 기온과 아메리카노 산점도
df.plot(kind="scatter", x="temp", y="americano",
        xlabel="최고기온(℃)",
        ylabel="아메리카노(잔)")